# Resume Screening AI - Experimentation Notebook

This notebook contains experiments and analysis for the resume screening AI project.

In [ ]:
# Import necessary libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import classification_report, confusion_matrix
import sys
import os

# Add parent directory to path
sys.path.append(os.path.join(os.path.dirname(os.getcwd()), '..'))

# Import project modules
from utils.preprocessing import load_and_preprocess_data, TextPreprocessor
from utils.feature_engineering import FeatureExtractor
from utils.training import ResumeJobMatcher, train_and_save_model
from utils.evaluation import ModelEvaluator
from utils.ranking import ResumeRanker

print("Libraries imported successfully!")

## 1. Data Loading and Preprocessing

In [ ]:
# Load and preprocess data
resumes_df, jobs_df = load_and_preprocess_data('../data/resumes.csv', '../data/jobs.csv')

print(f"Loaded {len(resumes_df)} resumes and {len(jobs_df)} jobs")
print("\nResume sample:")
display(resumes_df.head(2))
print("\nJob sample:")
display(jobs_df.head(2))

## 2. Exploratory Data Analysis

In [ ]:
# Experience distribution
plt.figure(figsize=(12, 4))

plt.subplot(1, 2, 1)
plt.hist(resumes_df['experience_years'], bins=10, alpha=0.7, color='skyblue')
plt.title('Candidate Experience Distribution')
plt.xlabel('Years of Experience')
plt.ylabel('Number of Candidates')

plt.subplot(1, 2, 2)
plt.hist(jobs_df['experience_required'], bins=10, alpha=0.7, color='lightcoral')
plt.title('Job Experience Requirements')
plt.xlabel('Required Experience (Years)')
plt.ylabel('Number of Jobs')

plt.tight_layout()
plt.show()

In [ ]:
# Skills analysis
all_resume_skills = ', '.join(resumes_df['skills']).split(', ')
all_job_skills = ', '.join(jobs_df['required_skills']).split(', ')

resume_skill_counts = pd.Series([skill.strip() for skill in all_resume_skills]).value_counts()
job_skill_counts = pd.Series([skill.strip() for skill in all_job_skills]).value_counts()

plt.figure(figsize=(15, 5))

plt.subplot(1, 2, 1)
resume_skill_counts.head(10).plot(kind='bar', color='lightblue')
plt.title('Top 10 Skills in Resumes')
plt.xticks(rotation=45)

plt.subplot(1, 2, 2)
job_skill_counts.head(10).plot(kind='bar', color='lightcoral')
plt.title('Top 10 Skills Required in Jobs')
plt.xticks(rotation=45)

plt.tight_layout()
plt.show()

print(f"Total unique skills in resumes: {len(resume_skill_counts)}")
print(f"Total unique skills required in jobs: {len(job_skill_counts)}")

## 3. Feature Engineering

In [ ]:
# Extract features
extractor = FeatureExtractor()
features = extractor.extract_all_features(resumes_df, jobs_df)

print("Feature extraction completed!")
print(f"Similarity matrix shape: {features['similarity_matrix'].shape}")
print(f"Skill match matrix shape: {features['skill_match_matrix'].shape}")
print(f"Experience match matrix shape: {features['experience_match_matrix'].shape}")

In [ ]:
# Analyze similarity scores
similarity_scores = features['similarity_matrix'].flatten()
skill_scores = features['skill_match_matrix'].flatten()
experience_scores = features['experience_match_matrix'].flatten()

plt.figure(figsize=(15, 4))

plt.subplot(1, 3, 1)
plt.hist(similarity_scores, bins=20, alpha=0.7, color='skyblue')
plt.title('Semantic Similarity Distribution')
plt.xlabel('Similarity Score')
plt.ylabel('Frequency')

plt.subplot(1, 3, 2)
plt.hist(skill_scores, bins=20, alpha=0.7, color='lightgreen')
plt.title('Skill Match Distribution')
plt.xlabel('Skill Match Score')

plt.subplot(1, 3, 3)
plt.hist(experience_scores, bins=20, alpha=0.7, color='lightcoral')
plt.title('Experience Match Distribution')
plt.xlabel('Experience Match Score')

plt.tight_layout()
plt.show()

print(f"Semantic similarity - Mean: {np.mean(similarity_scores):.3f}, Std: {np.std(similarity_scores):.3f}")
print(f"Skill match - Mean: {np.mean(skill_scores):.3f}, Std: {np.std(skill_scores):.3f}")
print(f"Experience match - Mean: {np.mean(experience_scores):.3f}, Std: {np.std(experience_scores):.3f}")

## 4. Model Training

In [ ]:
# Train models
training_results = train_and_save_model(features)

print("Model training completed!")
for model_name, result in training_results.items():
    print(f"\n{model_name.upper()}:")
    print(f"  Best params: {result['best_params']}")
    print(f"  Test score: {result['test_score']:.4f}")

## 5. Model Evaluation

In [ ]:
# Evaluate models
evaluator = ModelEvaluator()

# Get training data for evaluation
matcher = ResumeJobMatcher()
X, y = matcher.prepare_training_data(features)

# Split for evaluation
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

X_train_scaled = matcher.scaler.fit_transform(X_train)
X_test_scaled = matcher.scaler.transform(X_test)

# Evaluate each model
for model_name, model in matcher.models.items():
    model.fit(X_train_scaled, y_train)
    y_pred = model.predict(X_test_scaled)
    y_prob = model.predict_proba(X_test_scaled)[:, 1]
    
    results = evaluator.evaluate_model(y_test, y_pred, y_prob, model_name)
    
    print(f"\n{model_name.upper()} Results:")
    print(f"  Accuracy: {results['basic_metrics']['accuracy']:.4f}")
    print(f"  F1 Score: {results['basic_metrics']['f1_score']:.4f}")
    print(f"  AUC-ROC: {results['advanced_metrics']['auc_roc']:.4f}")

In [ ]:
# Model comparison
comparison_df = evaluator.compare_models()
display(comparison_df)

# Plot comparison
comparison_df.plot(kind='bar', figsize=(12, 6))
plt.title('Model Performance Comparison')
plt.xticks(rotation=45)
plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left')
plt.tight_layout()
plt.show()

## 6. Resume Ranking Analysis

In [ ]:
# Test ranking functionality
ranker = ResumeRanker('../models/model.pkl', '../models/scaler.pkl')

if ranker.is_loaded:
    # Test ranking for a sample job
    sample_job = jobs_df.iloc[0]
    rankings = ranker.rank_resumes_for_job(resumes_df, sample_job)
    
    print(f"Top 5 candidates for: {sample_job['job_title']}")
    display(rankings.head(5))
    
    # Get ranking summary
    summary = ranker.get_ranking_summary(rankings)
    print(f"\nRanking Summary:")
    print(f"  Total candidates: {summary['total_candidates']}")
    print(f"  Average score: {summary['average_score']:.3f}")
    print(f"  Score distribution: {summary['score_distribution']}")
else:
    print("Model not loaded. Please train the model first.")

In [ ]:
# Analyze ranking quality
if ranker.is_loaded:
    # Test ranking for multiple jobs
    ranking_qualities = []
    
    for _, job in jobs_df.iterrows():
        rankings = ranker.rank_resumes_for_job(resumes_df, job)
        quality = ranker.evaluate_ranking_quality(rankings)
        quality['job_title'] = job['job_title']
        ranking_qualities.append(quality)
    
    quality_df = pd.DataFrame(ranking_qualities)
    display(quality_df)
    
    # Average ranking metrics
    avg_metrics = quality_df.select_dtypes(include=[np.number]).mean()
    print("\nAverage Ranking Metrics:")
    display(avg_metrics)

## 7. Feature Importance Analysis

In [ ]:
# Get feature importance from the best model
if ranker.is_loaded:
    importance = ranker.get_feature_importance()
    
    if importance:
        print("Feature Importance:")
        for feature, score in importance.items():
            print(f"  {feature}: {score:.4f}")
        
        # Plot feature importance
        plt.figure(figsize=(8, 5))
        plt.bar(importance.keys(), importance.values(), color='skyblue')
        plt.title('Feature Importance')
        plt.ylabel('Importance Score')
        plt.xticks(rotation=45)
        plt.tight_layout()
        plt.show()
    else:
        print("Feature importance not available for this model type.")

## 8. Error Analysis

In [ ]:
# Analyze prediction errors
if ranker.is_loaded:
    # Get predictions for all resume-job pairs
    all_predictions = []
    
    for _, job in jobs_df.iterrows():
        rankings = ranker.rank_resumes_for_job(resumes_df, job)
        for _, ranking in rankings.iterrows():
            all_predictions.append({
                'job_title': job['job_title'],
                'candidate_name': ranking['candidate_name'],
                'composite_score': ranking['composite_score'],
                'skill_match': ranking['skill_match'],
                'experience_match': ranking['experience_match']
            })
    
    predictions_df = pd.DataFrame(all_predictions)
    
    # Find low-score matches (potential errors)
    low_score_matches = predictions_df[predictions_df['composite_score'] < 0.3]
    high_score_matches = predictions_df[predictions_df['composite_score'] > 0.8]
    
    print(f"Total predictions: {len(predictions_df)}")
    print(f"Low score matches (< 0.3): {len(low_score_matches)}")
    print(f"High score matches (> 0.8): {len(high_score_matches)}")
    
    # Analyze characteristics of low vs high score matches
    fig, axes = plt.subplots(1, 3, figsize=(15, 4))
    
    # Composite score distribution
    axes[0].hist(predictions_df['composite_score'], bins=20, alpha=0.7, color='skyblue')
    axes[0].set_title('Composite Score Distribution')
    axes[0].set_xlabel('Score')
    
    # Skill match vs composite score
    axes[1].scatter(predictions_df['skill_match'], predictions_df['composite_score'], alpha=0.5)
    axes[1].set_title('Skill Match vs Composite Score')
    axes[1].set_xlabel('Skill Match')
    axes[1].set_ylabel('Composite Score')
    
    # Experience match vs composite score
    axes[2].scatter(predictions_df['experience_match'], predictions_df['composite_score'], alpha=0.5)
    axes[2].set_title('Experience Match vs Composite Score')
    axes[2].set_xlabel('Experience Match')
    axes[2].set_ylabel('Composite Score')
    
    plt.tight_layout()
    plt.show()

## 9. Hyperparameter Tuning Experiments

In [ ]:
# Experiment with different weight combinations for composite score
if ranker.is_loaded:
    weight_combinations = [
        {'semantic': 0.4, 'skill': 0.4, 'experience': 0.2},  # Default
        {'semantic': 0.5, 'skill': 0.3, 'experience': 0.2},  # More semantic
        {'semantic': 0.3, 'skill': 0.5, 'experience': 0.2},  # More skill
        {'semantic': 0.3, 'skill': 0.3, 'experience': 0.4},  # More experience
        {'semantic': 0.6, 'skill': 0.2, 'experience': 0.2},  # Heavy semantic
    ]
    
    sample_job = jobs_df.iloc[0]
    
    weight_results = []
    
    for weights in weight_combinations:
        rankings = ranker.rank_resumes_for_job(resumes_df, sample_job, weights)
        avg_score = rankings['composite_score'].mean()
        std_score = rankings['composite_score'].std()
        top_3_avg = rankings.head(3)['composite_score'].mean()
        
        weight_results.append({
            'weights': weights,
            'avg_score': avg_score,
            'std_score': std_score,
            'top_3_avg': top_3_avg
        })
    
    weight_df = pd.DataFrame(weight_results)
    display(weight_df)
    
    # Plot weight comparison
    fig, axes = plt.subplots(1, 3, figsize=(15, 4))
    
    weight_labels = [f"S:{w['semantic']},K:{w['skill']},E:{w['experience']}" for w in weight_combinations]
    
    axes[0].bar(weight_labels, weight_df['avg_score'], color='skyblue')
    axes[0].set_title('Average Score by Weight Combination')
    axes[0].tick_params(axis='x', rotation=45)
    
    axes[1].bar(weight_labels, weight_df['std_score'], color='lightgreen')
    axes[1].set_title('Score Standard Deviation')
    axes[1].tick_params(axis='x', rotation=45)
    
    axes[2].bar(weight_labels, weight_df['top_3_avg'], color='lightcoral')
    axes[2].set_title('Top 3 Average Score')
    axes[2].tick_params(axis='x', rotation=45)
    
    plt.tight_layout()
    plt.show()

## 10. Conclusions and Insights

In [ ]:
# Summary statistics
print("=== EXPERIMENTATION SUMMARY ===")
print(f"Dataset: {len(resumes_df)} resumes, {len(jobs_df)} jobs")
print(f"Average candidate experience: {resumes_df['experience_years'].mean():.1f} years")
print(f"Average job requirement: {jobs_df['experience_required'].mean():.1f} years")

if ranker.is_loaded:
    print(f"\nModel Performance:")
    if not comparison_df.empty:
        best_model = comparison_df['f1_score'].idxmax()
        print(f"Best model: {best_model}")
        print(f"Best F1 score: {comparison_df.loc[best_model, 'f1_score']:.4f}")
    
    print(f"\nRanking Quality:")
    if 'avg_metrics' in locals():
        print(f"Average precision@5: {avg_metrics.get('precision@5', 'N/A'):.3f}")
        print(f"Average recall@5: {avg_metrics.get('recall@5', 'N/A'):.3f}")

print("\n=== KEY INSIGHTS ===")
print("1. Semantic similarity provides good baseline matching")
print("2. Skill matching is crucial for accurate rankings")
print("3. Experience requirements should be weighted appropriately")
print("4. Model ensemble approach improves overall performance")
print("5. Feature engineering significantly impacts matching quality")